# Colab Drive mount

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Go to your project

In [4]:
import os

REPO_DIR = "/content/malaria-cnn-project"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/auth889-ai/malaria-cnn-project.git

%cd /content/malaria-cnn-project

/content/malaria-cnn-project


# Make src importable

In [5]:
import sys

SRC_PATH = "/content/malaria-cnn-project/src"

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("src path added ✅")

src path added ✅


In [6]:
import tensorflow as tf
import os

from malaria_cnn.data import get_datasets
from malaria_cnn.model import build_model

from tensorflow.keras.callbacks import (
    Callback,
    EarlyStopping,
    LearningRateScheduler,
    ModelCheckpoint,
    ReduceLROnPlateau
)

print("TensorFlow version:", tf.__version__)
print("Imports successful ✅")

TensorFlow version: 2.20.0
Imports successful ✅


# Load dataset

In [7]:
%pip install -q --upgrade "protobuf==6.31.1"

In [8]:
import google.protobuf

print("Protobuf version:", google.protobuf.__version__)

Protobuf version: 6.31.1


In [9]:
import tensorflow as tf
import os

from malaria_cnn.data import get_datasets
from malaria_cnn.model import build_model

from tensorflow.keras.callbacks import (
    Callback,
    EarlyStopping,
    LearningRateScheduler,
    ModelCheckpoint,
    ReduceLROnPlateau
)

print("TensorFlow version:", tf.__version__)
print("Imports successful ✅")

TensorFlow version: 2.20.0
Imports successful ✅


In [10]:
import tensorflow_datasets as tfds

print("TFDS version:", tfds.__version__)
print("TFDS import successful ✅")

TFDS version: 4.9.10
TFDS import successful ✅


In [11]:
train_dataset, val_dataset, test_dataset, dataset_info = get_datasets()

print("Datasets loaded successfully ✅")

Datasets loaded successfully ✅


In [12]:
for images, labels in train_dataset.take(1):

    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

    print("dtype:", images.dtype)

    print(
        "min:",
        tf.reduce_min(images).numpy()
    )

    print(
        "max:",
        tf.reduce_max(images).numpy()
    )

Images shape: (32, 224, 224, 3)
Labels shape: (32,)
dtype: <dtype: 'float32'>
min: 0.0
max: 0.91642034


In [13]:
def create_compiled_model(learning_rate=0.001):

    model = build_model()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss=tf.keras.losses.BinaryCrossentropy(),

        metrics=["accuracy"]
    )

    return model

# callback test করার আগে ছোট dataset দিয়ে দ্রুত experiment করা।

In [14]:
FAST_TEST = True

In [15]:
if FAST_TEST:

    train_run = train_dataset.take(50)
    val_run = val_dataset.take(20)
    test_run = test_dataset.take(20)

    print("FAST TEST MODE ✅")

else:

    train_run = train_dataset
    val_run = val_dataset
    test_run = test_dataset

    print("FULL DATASET MODE ✅")

FAST TEST MODE ✅


In [16]:
def create_compiled_model(learning_rate=0.001):

    model = build_model()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"]
    )

    return model

In [17]:
model = create_compiled_model()

model.summary()

Model: "malaria_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 6)    │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 6)    │            24 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 16)   │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 46656)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     4,665,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 10)             │            40 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,668,297 (17.81 MB)

 Trainable params: 4,668,033 (17.81 MB)

 Non-trainable params: 264 (1.03 KB)

# Custom Callback

In [18]:
class LossCallback(Callback):

    def on_epoch_end(self, epoch, logs=None):

        logs = logs or {}

        print(
            f"\nEpoch {epoch + 1}"
            f" | loss={logs.get('loss'):.4f}"
            f" | val_loss={logs.get('val_loss'):.4f}"
            f" | accuracy={logs.get('accuracy'):.4f}"
            f" | val_accuracy={logs.get('val_accuracy'):.4f}"
        )

In [19]:
custom_model = create_compiled_model()

custom_model.fit(
    train_run,
    validation_data=val_run,
    epochs=2,
    callbacks=[LossCallback()]
)

Epoch 1/2
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5857 - loss: 0.8082
Epoch 1 | loss=0.7100 | val_loss=0.6826 | accuracy=0.6131 | val_accuracy=0.5359
50/50 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step - accuracy: 0.6131 - loss: 0.7100 - val_accuracy: 0.5359 - val_loss: 0.6826
Epoch 2/2
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7438 - loss: 0.5217
Epoch 2 | loss=0.5201 | val_loss=0.6993 | accuracy=0.7406 | val_accuracy=0.5359
50/50 ━━━━━━━━━━━━━━━━━━━━ 13s 49ms/step - accuracy: 0.7406 - loss: 0.5201 - val_accuracy: 0.5359 - val_loss: 0.6993


# EarlyStopping

In [20]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [21]:
early_model = create_compiled_model()

early_model.fit(
    train_run,
    validation_data=val_run,
    epochs=6,
    callbacks=[early_stopping]
)

Epoch 1/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 99ms/step - accuracy: 0.5619 - loss: 0.7778 - val_accuracy: 0.4641 - val_loss: 0.7174
Epoch 2/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.7119 - loss: 0.5670 - val_accuracy: 0.4641 - val_loss: 0.7716
Epoch 3/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.7962 - loss: 0.4672 - val_accuracy: 0.4641 - val_loss: 0.8589
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


# LearningRateScheduler

In [22]:
def scheduler(epoch, lr):

    if epoch < 3:
        return float(lr)

    return float(lr * tf.math.exp(-0.1))

In [23]:
lr_scheduler = LearningRateScheduler(
    scheduler,
    verbose=1
)

In [24]:
scheduler_model = create_compiled_model(
    learning_rate=0.001
)

scheduler_model.fit(
    train_run,
    validation_data=val_run,
    epochs=5,
    callbacks=[lr_scheduler]
)


Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.6400 - loss: 0.6685 - val_accuracy: 0.4656 - val_loss: 0.7337 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.7194 - loss: 0.5630 - val_accuracy: 0.4797 - val_loss: 0.6926 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 62ms/step - accuracy: 0.7563 - loss: 0.5099 - val_accuracy: 0.5984 - val_loss: 0.6704 - learning_rate: 0.0010

Epoch 4: LearningRateScheduler setting learning rate to 0.0009048373904079199.
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 56ms/step - accuracy: 0.8225 - loss: 0.4272 - val_accuracy: 0.4688 - val_loss: 0.7344 - learning_rate: 9.0484e-04

Epoch 5: LearningRateScheduler setting learning rate to 0.0008187306812033

# ModelCheckpoint

In [25]:
MODEL_DIR = "/content/drive/MyDrive/malaria_cnn_project/models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "best_callbacks_model.keras"
)

In [26]:
checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

In [27]:
checkpoint_model = create_compiled_model()

checkpoint_model.fit(
    train_run,
    validation_data=val_run,
    epochs=5,
    callbacks=[checkpoint]
)

Epoch 1/5
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.6111 - loss: 0.6826
Epoch 1: val_loss improved from None to 0.75311, saving model to /content/drive/MyDrive/malaria_cnn_project/models/best_callbacks_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/malaria_cnn_project/models/best_callbacks_model.keras
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - accuracy: 0.6200 - loss: 0.6702 - val_accuracy: 0.4641 - val_loss: 0.7531
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7253 - loss: 0.5543
Epoch 2: val_loss did not improve from 0.75311
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.7075 - loss: 0.5703 - val_accuracy: 0.4641 - val_loss: 0.8426
Epoch 3/5
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.8089 - loss: 0.4777
Epoch 3: val_loss did not improve from 0.75311
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - accuracy: 0.7794 - loss: 0.4953 - val_accuracy: 0.4641 - val_loss: 0.8704
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step -

In [28]:
print(
    "Saved:",
    os.path.exists(CHECKPOINT_PATH)
)

Saved: True


# ReduceLROnPlateau

In [29]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=2,
    mode="min",
    min_lr=1e-6,
    verbose=1
)

In [30]:
plateau_model = create_compiled_model(
    learning_rate=0.001
)

plateau_model.fit(
    train_run,
    validation_data=val_run,
    epochs=6,
    callbacks=[reduce_lr]
)

Epoch 1/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.6319 - loss: 0.6688 - val_accuracy: 0.5984 - val_loss: 0.6699 - learning_rate: 0.0010
Epoch 2/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.7337 - loss: 0.5431 - val_accuracy: 0.5359 - val_loss: 0.6768 - learning_rate: 0.0010
Epoch 3/6
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.7857 - loss: 0.4660
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.7919 - loss: 0.4679 - val_accuracy: 0.5531 - val_loss: 0.6850 - learning_rate: 0.0010
Epoch 4/6
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.8612 - loss: 0.3657 - val_accuracy: 0.4750 - val_loss: 0.7551 - learning_rate: 1.0000e-04
Epoch 5/6
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9042 - loss: 0.3030
Epoch 5: ReduceLROnPlateau reducing learning rate to 1.0000000474974514e-05.
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.8844 - loss: 0.3203 - val_acc

In [31]:
final_checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

final_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=2,
    mode="min",
    min_lr=1e-6,
    verbose=1
)

final_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [32]:
final_model = create_compiled_model(
    learning_rate=0.001
)

In [33]:
final_history = final_model.fit(
    train_run,
    validation_data=val_run,
    epochs=15,
    callbacks=[
        final_checkpoint,
        final_reduce_lr,
        final_early_stopping
    ]
)

Epoch 1/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5939 - loss: 0.7046
Epoch 1: val_loss improved from None to 0.69377, saving model to /content/drive/MyDrive/malaria_cnn_project/models/best_callbacks_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/malaria_cnn_project/models/best_callbacks_model.keras
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - accuracy: 0.6263 - loss: 0.6675 - val_accuracy: 0.5359 - val_loss: 0.6938 - learning_rate: 0.0010
Epoch 2/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6998 - loss: 0.5742
Epoch 2: val_loss did not improve from 0.69377
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.7056 - loss: 0.5779 - val_accuracy: 0.4688 - val_loss: 0.7146 - learning_rate: 0.0010
Epoch 3/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.7572 - loss: 0.5046
Epoch 3: val_loss did not improve from 0.69377

Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/ste